# Agent 第 01 课：什么是 Agent，和直接调 LLM 有什么区别

这门课和 RAG 系列一个思路：**前期不借助任何框架**（没有 LangChain / LlamaIndex / smolagents），我们用最朴素的 Python 和 OpenAI 兼容 API 把原理搞清楚。框架是后期的事。

先定义清楚一个最容易混淆的问题。

## 一句话定义

> **LLM Agent = LLM + 动作空间（工具）+ 循环。**

普通 LLM 调用是一次性的：你问一句，它答一句，结束。Agent 不一样——它可以**多轮地、自主地**：

1. 看到当前任务和已有信息
2. **决定**下一步做什么（回答、还是调用某个工具）
3. 如果是调用工具，**执行**工具，拿到结果
4. 把结果喂回自己，继续从第 1 步来，直到觉得任务完成

也就是说：**LLM 负责"想"，外部代码负责"做"和"循环"**。LLM 本身一次 forward pass 是纯文本函数，它并没有"agent"特性——是你写的那段循环让它变成了 agent。

这是这门课最重要的一句话，请记住。

## 一个具体对比

**任务**："123 × 456 是多少？"

### 方式 A：直接问 LLM（不是 agent）
```
User:  123 × 456 是多少？
LLM:   56088（←它在靠"算术直觉"给你，偶尔会错）
```

### 方式 B：Agent（带计算器工具）
```
User:  123 × 456 是多少？
LLM:   我应该用计算器。Action: calculator("123*456")
Tool:  56088
LLM:   答案是 56088。
```

区别不在于最终答案是否正确，而在于：

- 方式 A 的"知识"和"能力"被锁在模型权重里
- 方式 B 可以**插入外部能力**（计算、搜索、调 API、读文件、查数据库、写代码并运行……）

当任务复杂到一个 prompt 答不完时，agent 的循环能力会带来指数级差距。

## ReAct：最经典的 Agent 循环

2022 年论文 *ReAct: Synergizing Reasoning and Acting in Language Models* 提出了一个非常简单但极其有效的范式，后来几乎所有框架都是它的变体：

```
Thought:       我该怎么做？（LLM 的推理）
Action:        调用哪个工具，参数是什么？（LLM 产出）
Observation:   工具执行的结果（外部代码返回）
Thought:       基于观察，下一步该怎么办？
Action:        ...
Observation:   ...
...
Final Answer:  给用户的最终回答
```

**关键洞察**：把"推理"(Thought) 和"行动"(Action) 交织起来，让模型每一步都**先说服自己为什么这么做**，再行动。对比：

- 只 Reasoning（只让它想）→ 想得很好但不落地
- 只 Acting（只让它调工具）→ 经常盲目乱调
- **ReAct** → 两者互相约束，错误率显著下降

后面的 Plan-and-Execute、Reflection、Tree-of-Thoughts 都可以看作 ReAct 的演化。我们这一课先用最简单的形式把它跑起来——**不涉及工具**，只让模型输出 Thought 和 Final Answer 两种结构，把"循环"的骨架搭出来。

## 连接本地 LLM

In [ ]:
from openai import OpenAI

client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
preferred = ['qwen/qwen3.6-35b-a3b', 'qwen3.6-35b-a3b', 'qwen3.5-35b-a3b']
chat_model = next((m for m in preferred if m in model_ids), model_ids[0])
print('chat model =', chat_model)
print('all models:', model_ids)

## 一个最朴素的"agent 骨架"（还没有工具）

我们先让 LLM 按一个**结构化的格式**输出：

```
Thought: <我的思考>
Final Answer: <给用户的答案>
```

然后写一个循环来解析它。这看起来像杀鸡用牛刀（反正只有一轮），但这是在搭架构——下一课加上工具的时候，这个架构就直接派上用场了。

**为什么要从结构化输出开始**：agent 的本质就是"让 LLM 的输出能被下游代码理解"。如果 LLM 只会输出散文，你写的任何循环都没有抓手。

In [ ]:
import re

SYSTEM_PROMPT = """You are a reasoning agent. Always respond in EXACTLY this format:

Thought: <your reasoning in one or two sentences>
Final Answer: <your answer to the user>

Do not output anything else. No JSON, no markdown, just those two lines."""

def call_llm(messages, temperature=0):
    r = client.chat.completions.create(
        model=chat_model, temperature=temperature, messages=messages
    )
    return r.choices[0].message.content.strip()

def parse_output(text):
    # 非常朴素的解析——后面会慢慢改良
    thought_m = re.search(r'Thought:\s*(.+?)(?=\n[A-Z]|\Z)', text, flags=re.S)
    answer_m  = re.search(r'Final Answer:\s*(.+?)\Z', text, flags=re.S)
    return {
        'thought': thought_m.group(1).strip() if thought_m else None,
        'final_answer': answer_m.group(1).strip() if answer_m else None,
        'raw': text,
    }

def run_agent(user_question, max_steps=5):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_question},
    ]
    for step in range(max_steps):
        text = call_llm(messages)
        parsed = parse_output(text)
        print(f'--- step {step+1} ---')
        print(parsed['raw'])
        print()
        if parsed['final_answer'] is not None:
            return parsed['final_answer']
        # 如果没拿到 Final Answer，就把模型输出塞回去让它继续（这一课用不到，占位）
        messages.append({'role': 'assistant', 'content': text})
        messages.append({'role': 'user', 'content': 'Please provide a Final Answer now.'})
    return None

final = run_agent('用一句话解释什么是量子纠缠。')
print('>>> final answer:', final)

## 看看骨架里已经有什么了

虽然上面这段代码还不是真正的 agent（没工具、没多步），但请仔细看——**agent 所有核心元素的雏形都在这了**：

| 元素 | 代码里的位置 |
|---|---|
| 系统角色设定 | `SYSTEM_PROMPT` |
| 结构化输出协议 | `Thought:` / `Final Answer:` |
| 解析器 | `parse_output` |
| 循环 | `for step in range(max_steps)` |
| 对话历史 | `messages` 列表 |
| 终止条件 | 拿到 `Final Answer` |
| 保险丝 | `max_steps`（不让它无限转） |

下一课我们只加一件事——**Action**，也就是工具调用——骨架就会立刻变成一个能干活的 agent。

## 本课工程直觉

1. **Agent 的魔法不在 LLM 里，在你写的那段循环里**。LLM 本身就是一次函数调用，循环让它有了"持续工作"的能力。
2. **结构化输出 + 解析器是一切 agent 的根基**。模型要么输出下一步动作，要么输出终止信号，下游代码必须能区分。
3. **永远要有 `max_steps` 保险丝**。模型会陷入死循环，你的代码要能兜底。
4. **先写协议，再写能力**。别一上来就接一堆工具，先把"agent 怎么和外部世界说话"的协议定义好。
5. 这一课的 `Thought: / Final Answer:` 格式很脆弱（换个模型可能就不听话了），下一课我们会升级到 **JSON 输出**，那才是生产级 agent 用的协议。

---

下一课：**第 02 课 Tool Use 的底层协议**——我们会把"动作"正式引入，并且讲清楚 OpenAI function calling / JSON schema 这套工业标准到底是怎么回事，以及它和我们今天这个字符串协议的优劣。